# DeepDream con rete a scelta

**Come usarlo**
1. Menu *Runtime → Cambia tipo di runtime → GPU*.
2. Esegui le celle **in ordine** (le prime tre definiscono il codice, ci vuole poco).
3. Cambia i parametri nella cella *Parametri*, poi riesegui solo *Genera* (e, se vuoi, *Zoom ricorsivo*).

Per partire da una tua foto, caricala dal pannello file a sinistra e metti il suo nome nel parametro `immagine`.

In [ ]:
import random
import torch
import torch.nn.functional as F
import torchvision.models as models
from PIL import Image
from torchvision.transforms.functional import to_pil_image, to_tensor
from IPython.display import display, Image as IPyImage

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)

In [ ]:
class StopForward(Exception):
    # Interrompe il forward appena letto lo strato scelto
    pass


class Dreamer:
    def __init__(self, model_name="vgg19"):
        self.name = model_name
        self.model = getattr(models, model_name)(weights="DEFAULT").eval().to(device)
        for p in self.model.parameters():
            p.requires_grad_(False)   # i pesi restano fermi: cambia solo l'immagine
        self.mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
        self.handle = None
        self.store = {}
        self.channel = None

    def list_layers(self):
        for name, m in self.model.named_modules():
            if name and name.count(".") <= 1:
                print(f"{name:30s} {m.__class__.__name__}")

    def set_layer(self, layer, channel=None):
        if self.handle is not None:
            self.handle.remove()
        modules = dict(self.model.named_modules())
        if layer not in modules:
            raise ValueError(f"Strato '{layer}' non trovato: usa d.list_layers()")

        def hook(_module, _inp, out):
            self.store["act"] = out
            raise StopForward

        self.handle = modules[layer].register_forward_hook(hook)
        self.channel = channel

    def objective(self, x):
        # L(x) = 1/2 * mean(a^2), su tutto lo strato o su un solo canale
        try:
            self.model((x - self.mean) / self.std)
        except StopForward:
            pass
        a = self.store["act"]
        if self.channel is not None:
            a = a[:, self.channel]
        return 0.5 * (a ** 2).mean()

In [ ]:
def symmetrize(x, mode):
    # Forza la simmetria (serve un'immagine quadrata per la rotazione)
    if mode in ("speculare", "entrambe"):
        x = (x + torch.flip(x, [3])) / 2
    if mode in ("rotazionale", "entrambe"):
        x = sum(torch.rot90(x, k, (2, 3)) for k in range(4)) / 4
    return x


def dream(d, x, steps, lr, jitter, symmetry):
    # Gradient ascent sui pixel: x <- x + eta * g / mean|g|
    for _ in range(steps):
        ox = random.randint(-jitter, jitter) if jitter else 0
        oy = random.randint(-jitter, jitter) if jitter else 0
        xs = torch.roll(x, shifts=(ox, oy), dims=(2, 3)).detach().requires_grad_(True)
        (g,) = torch.autograd.grad(d.objective(xs), xs)
        g = torch.roll(g, shifts=(-ox, -oy), dims=(2, 3))
        g = g / (g.abs().mean() + 1e-8)
        x = (x + lr * g).clamp(0, 1)
        x = symmetrize(x, symmetry).detach()
    return x


def dream_octaves(d, base, octaves, scale, steps, lr, jitter, symmetry):
    H, W = base.shape[2:]
    sizes = [(int(H / scale ** i), int(W / scale ** i)) for i in reversed(range(octaves))]
    detail = None
    for k, (h, w) in enumerate(sizes, start=1):
        img_o = F.interpolate(base, size=(h, w), mode="bilinear", align_corners=False)
        if detail is None:
            x = img_o
        else:
            x = img_o + F.interpolate(detail, size=(h, w), mode="bilinear", align_corners=False)
        x = dream(d, x.clamp(0, 1), steps, lr, jitter, symmetry)
        detail = x - img_o
    return x


def make_start(start, size, image_path=None, square=False):
    if start == "rumore":
        x = 0.5 + 0.1 * torch.randn(1, 3, size, size)
    else:
        img = Image.open(image_path).convert("RGB")
        if square:   # ritaglio quadrato centrale
            w, h = img.size
            s = min(w, h)
            left, top = (w - s) // 2, (h - s) // 2
            img = img.crop((left, top, left + s, top + s))
        img.thumbnail((size, size))
        x = to_tensor(img).unsqueeze(0)
    return x.clamp(0, 1).to(device)

In [ ]:
#@title Parametri
modello = "vgg19" #@param ["vgg19", "vgg16", "googlenet", "resnet50", "alexnet"]
strato = "features.20" #@param {type:"string"}
canale = -1 #@param {type:"integer"}
partenza = "rumore" #@param ["rumore", "immagine"]
immagine = "foto.jpg" #@param {type:"string"}
simmetria = "rotazionale" #@param ["nessuna", "speculare", "rotazionale", "entrambe"]
dimensione = 512 #@param {type:"integer"}
ottave = 4 #@param {type:"integer"}
scala_ottave = 1.4 #@param {type:"number"}
passi = 20 #@param {type:"integer"}
lr = 0.01 #@param {type:"number"}
jitter = 8 #@param {type:"integer"}

# canale = -1 significa "tutto lo strato".
# Per vedere gli strati di una rete: d.list_layers() (dopo aver eseguito la cella Genera).
# Nomi degli strati: vgg -> features.N ; googlenet -> inception4c ; resnet50 -> layer3

In [ ]:
#@title Genera
if "d" not in globals() or d.name != modello:
    d = Dreamer(modello)
d.set_layer(strato, None if canale < 0 else canale)

base = make_start(partenza, dimensione, immagine, square=(simmetria != "nessuna"))
result = dream_octaves(d, base, ottave, scala_ottave, passi, lr, jitter, simmetria)

out = to_pil_image(result.cpu()[0].clamp(0, 1))
out.save("dream.png")
display(out)

In [ ]:
#@title Zoom ricorsivo (effetto frattale) - parte dall'ultimo risultato
n_frame = 20 #@param {type:"integer"}
zoom = 1.05 #@param {type:"number"}
passi_per_frame = 10 #@param {type:"integer"}

x = result.clone()
H, W = x.shape[2:]
frames = [to_pil_image(x.cpu()[0].clamp(0, 1))]

for i in range(n_frame):
    h, w = int(H / zoom), int(W / zoom)
    top, left = (H - h) // 2, (W - w) // 2
    crop = x[:, :, top:top + h, left:left + w]
    x = F.interpolate(crop, size=(H, W), mode="bilinear", align_corners=False)
    x = dream_octaves(d, x, 2, scala_ottave, passi_per_frame, lr, jitter, simmetria)
    frames.append(to_pil_image(x.cpu()[0].clamp(0, 1)))

frames[0].save("zoom.gif", save_all=True, append_images=frames[1:], duration=120, loop=0)
display(IPyImage("zoom.gif"))

In [ ]:
# Elenco degli strati della rete caricata (per scegliere il parametro "strato")
d.list_layers()